# 08 — Gemini Neutral Summary
Summarises only the **news metadata** (titles/sources/dates/urls). Saves prompt+response for provenance.

In [ ]:
import os, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import requests

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def mask(s: str, keep=4):
    if not s: return ""
    s = str(s)
    return s[:keep] + "…" + "*"*6 if len(s) > keep else "*"*len(s)

def env(name: str, default=""):
    v = os.getenv(name, default)
    return v.strip() if isinstance(v, str) else v

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": [],
    }

def add_artifact(manifest: dict, path: Path, kind: str, meta=None):
    meta = meta or {}
    manifest["artifacts"].append({
        "kind": kind,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "sha256": sha256_file(path) if path.exists() and path.is_file() else None,
        "meta": meta,
    })

print("PROJECT_ROOT:", PROJECT_ROOT)
print("UTC now:", utcnow())

In [ ]:
step = "08_gemini_neutral_summary"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

api_key = env("GEMINI_API_KEY")
model = env("GEMINI_MODEL", "gemini-1.5-flash")

news_path = OUTPUTS/"03_news_context"/"news_articles.json"
if not news_path.exists():
    raise FileNotFoundError("Run notebook 03 first: outputs/03_news_context/news_articles.json")

articles = load_json(news_path)

lines = []
for a in articles[:25]:
    lines.append(f"- {a.get('publishedAt')} | {a.get('source')} | {a.get('title')} | {a.get('url')}")
context = "\n".join(lines)

prompt = f"""You are assisting an environmental assessment project.
Write a neutral, non-sensational context summary based ONLY on the following news metadata lines.
Do not speculate. Do not claim causality. If evidence is insufficient, say so.

Return:
1) 5-bullet neutral summary
2) 5 candidate keywords for later searching
3) A short caveats paragraph

News metadata:
{context}
"""

write_json(out/"gemini_prompt.json", {"model": model, "prompt": prompt})

if not api_key:
    print("⚠️ GEMINI_API_KEY missing; prompt saved only.")
else:
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    payload = {"contents": [{"parts": [{"text": prompt}]}]}
    r = requests.post(url, params={"key": api_key}, json=payload, timeout=60)
    resp = r.json() if r.headers.get("content-type","").startswith("application/json") else {"status":"nonjson","text": r.text}
    write_json(out/"gemini_response.json", resp)
    print("Gemini HTTP:", r.status_code)

manifest = manifest_base(step, config_paths=[CONFIGS/"run.yml"])
add_artifact(manifest, out/"gemini_prompt.json", "prompt")
if (out/"gemini_response.json").exists():
    add_artifact(manifest, out/"gemini_response.json", "llm_response")
write_json(out/"manifest.json", manifest)
print("Wrote:", out/"manifest.json")